Очистка и подготовка данных:
1. Удалите дубликаты и неактуальные столбцы (неактуальные потом).
2. Соответствующим образом обработайте отсутствующие значения. (Этот пункт буду делать по мере необходимости в процессе анализа)
3. Преобразуйте типы данных для таких столбцов, как даты и числовые
значения.

In [27]:
import pandas as pd

# Загружаем Excel-файл
spend_df = pd.read_excel("Spend.xlsx")

# Выводим первые строки
print("🔹 Первые строки датафрейма:")
print(spend_df.head())

# Проверяем форматы данных (типы столбцов)
print("\n🔹 Информация о типах данных:")
print(spend_df.info())


🔹 Первые строки датафрейма:
        Date        Source               Campaign  Impressions  Spend  Clicks  \
0 2023-07-03    Google Ads         gen_analyst_DE            6   0.00       0   
1 2023-07-03    Google Ads  performancemax_eng_DE            4   0.01       1   
2 2023-07-03  Facebook Ads                    NaN            0   0.00       0   
3 2023-07-03    Google Ads                    NaN            0   0.00       0   
4 2023-07-03           CRM                    NaN            0   0.00       0   

  AdGroup   Ad  
0     NaN  NaN  
1     NaN  NaN  
2     NaN  NaN  
3     NaN  NaN  
4     NaN  NaN  

🔹 Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20779 entries, 0 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         20779 non-null  datetime64[ns]
 1   Source       20779 non-null  object        
 2   Campaign     14785 non-null  object        


In [2]:
num_duplicates = spend_df.duplicated().sum()
print(f"Количество полных дубликатов: {num_duplicates}")


Количество полных дубликатов: 917


In [3]:
# Удаляем полные дубликаты строк
spend_df = spend_df.drop_duplicates()

# Проверяем, сколько строк осталось после удаления
print(f"После удаления дубликатов осталось {len(spend_df)} строк.")


После удаления дубликатов осталось 19862 строк.


In [4]:
# Сохраняем очищенный датафрейм в новый Excel-файл
spend_df.to_excel("Spend_f_clean.xlsx", index=False)

print("✅ Файл успешно сохранён как 'Spend_f_clean.xlsx'")


✅ Файл успешно сохранён как 'Spend_f_clean.xlsx'


Теперь файл Deals

In [20]:
import pandas as pd

# Загружаем Excel-файл
deals_df = pd.read_excel("Deals.xlsx", dtype={"Id": str,"Contact Name": str})

# Выводим первые строки
print("🔹 Первые строки датафрейма:")
print(deals_df.head())

# Проверяем форматы данных (типы столбцов)
print("\n🔹 Информация о типах данных:")
print(deals_df.info())

🔹 Первые строки датафрейма:
                    Id Deal Owner Name Closing Date            Quality  \
0  5805028000056864695        Ben Hall          NaN                NaN   
1  5805028000056859489   Ulysses Adams          NaN                NaN   
2  5805028000056832357   Ulysses Adams   21.06.2024     D - Non Target   
3  5805028000056824246        Eva Kent   21.06.2024  E - Non Qualified   
4  5805028000056873292        Ben Hall   21.06.2024     D - Non Target   

      Stage     Lost Reason       Page                  Campaign       SLA  \
0  New Lead             NaN  /eng/test             03.07.23women       NaN   
1  New Lead             NaN    /at-eng                       NaN       NaN   
2      Lost      Non target    /at-eng                engwien_AT  00:26:43   
3      Lost  Invalid number       /eng  04.07.23recentlymoved_DE  01:00:04   
4      Lost      Non target       /eng              discovery_DE  00:53:12   

              Content  ...        Product Education Type  

In [21]:
import numpy as np

# Копируем, чтобы работать безопасно
deals_conv = deals_df.copy()


# 2️⃣ Closing Date → дата (дд.мм.гггг)
deals_conv["Closing Date"] = pd.to_datetime(
    deals_conv["Closing Date"],
    format="%d.%m.%Y",
    errors="coerce"
)


# 4️⃣ Created Time → дата и время (дд.мм.гггг чч:мм)
deals_conv["Created Time"] = pd.to_datetime(
    deals_conv["Created Time"],
    format="%d.%m.%Y %H:%M",
    errors="coerce"
)

# 5️⃣ Course duration → int64
deals_conv["Course duration"] = pd.to_numeric(
    deals_conv["Course duration"],
    errors="coerce"
).astype("Int64")  # чтобы поддерживать пропуски

# 6️⃣ Months of study → int64
deals_conv["Months of study"] = pd.to_numeric(
    deals_conv["Months of study"],
    errors="coerce"
).astype("Int64")

# 7️⃣ Initial Amount Paid → float64
# убираем возможные символы валюты или запятые
deals_conv["Initial Amount Paid"] = (
    deals_conv["Initial Amount Paid"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace("[^0-9.]", "", regex=True)
)
deals_conv["Initial Amount Paid"] = pd.to_numeric(
    deals_conv["Initial Amount Paid"], errors="coerce"
)

# 8️⃣ Offer Total Amount → float64
deals_conv["Offer Total Amount"] = (
    deals_conv["Offer Total Amount"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace("[^0-9.]", "", regex=True)
)
deals_conv["Offer Total Amount"] = pd.to_numeric(
    deals_conv["Offer Total Amount"], errors="coerce"
)



# Проверим результат
print("✅ Типы данных после преобразования:")
print(deals_conv.dtypes)


✅ Типы данных после преобразования:
Id                             object
Deal Owner Name                object
Closing Date           datetime64[ns]
Quality                        object
Stage                          object
Lost Reason                    object
Page                           object
Campaign                       object
SLA                            object
Content                        object
Term                           object
Source                         object
Payment Type                   object
Product                        object
Education Type                 object
Created Time           datetime64[ns]
Course duration                 Int64
Months of study                 Int64
Initial Amount Paid           float64
Offer Total Amount            float64
Contact Name                   object
City                           object
Level of Deutsch               object
dtype: object


In [22]:
# Приводим к строке и убираем лишние пробелы
deals_conv["SLA"] = deals_conv["SLA"].astype(str).str.strip()

# Создаём колонку, где будем хранить часы
sla_hours = []

for val in deals_conv["SLA"]:
    if pd.isna(val) or val in ["nan", "NaT"]:
        sla_hours.append(np.nan)
        continue

    # Если есть дата (формат dd.mm.yyyy HH:MM:SS)
    if "." in val and len(val.split()[0].split(".")) == 3:
        # Парсим datetime
        dt = pd.to_datetime(val, format="%d.%m.%Y %H:%M:%S", errors="coerce")
        if dt is pd.NaT:
            sla_hours.append(np.nan)
        else:
            # Excel дата 01.01.1900 → вычитаем 1899-12-31, чтобы посчитать дни + часы
            delta = dt - pd.Timestamp("1899-12-31")
            hours = delta.total_seconds() / 3600
            sla_hours.append(round(hours, 4))
    else:
        # Только время, например "19:49:18"
        t = pd.to_timedelta(val)
        hours = t.total_seconds() / 3600
        sla_hours.append(round(hours, 4))

# Сохраняем обратно
deals_conv["SLA_hours"] = sla_hours

# Проверим результат
deals_conv[["SLA", "SLA_hours"]].head(10)
print("Максимальное SLA (часы):", deals_conv["SLA_hours"].max())


Максимальное SLA (часы): 7474.5733


In [23]:
deals_conv[["SLA", "SLA_hours"]].head(10)

,SLA,SLA_hours
0,nan,NaN
1,nan,NaN
2,00:26:43,0.4453
3,01:00:04,1.0011
4,00:53:12,0.8867
5,01:33:10,1.5528
6,nan,NaN
7,02:12:29,2.2081
8,nan,NaN
9,00:10:08,0.1689


In [24]:
num_duplicates = deals_conv.duplicated().sum()
print(f"Количество полных дубликатов: {num_duplicates}")


Количество полных дубликатов: 0


В поле Lost Reason есть еще - Дубликаты, отмеченные вручную, удалим их тоже.

In [25]:
# Удаляем строки с Lost Reason == 'Duplicate'
deals_conv = deals_conv[deals_conv["Lost Reason"] != "Duplicate"]

# Проверим, сколько строк осталось
print(f"Количество строк после удаления дубликатов по Lost Reason: {len(deals_conv)}")



Количество строк после удаления дубликатов по Lost Reason: 19824


In [26]:
# Сохраняем очищенный датафрейм в новый Excel-файл
deals_conv.to_excel("Deals_f_clean.xlsx", index=False)

print("✅ Файл успешно сохранён как 'Deals_f_clean.xlsx'")


✅ Файл успешно сохранён как 'Deals_f_clean.xlsx'


Теперь файл Contacts

In [15]:
# Загружаем Excel-файл
contacts_df = pd.read_excel("Contacts.xlsx")

# Выводим первые строки
print("🔹 Первые строки датафрейма:")
print(contacts_df.head())

# Проверяем форматы данных (типы столбцов)
print("\n🔹 Информация о типах данных:")
print(contacts_df.info())

🔹 Первые строки датафрейма:
                    Id Contact Owner Name      Created Time     Modified Time
0  5805028000000645014       Rachel White  27.06.2023 11:28  22.12.2023 13:34
1  5805028000000872003      Charlie Davis  03.07.2023 11:31  21.05.2024 10:23
2  5805028000000889001          Bob Brown  02.07.2023 22:37  21.12.2023 13:17
3  5805028000000907006          Bob Brown  03.07.2023 05:44  29.12.2023 15:20
4  5805028000000939010         Nina Scott  04.07.2023 10:11  16.04.2024 16:14

🔹 Информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18548 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Id                  18548 non-null  int64 
 1   Contact Owner Name  18548 non-null  object
 2   Created Time        18548 non-null  object
 3   Modified Time       18548 non-null  object
dtypes: int64(1), object(3)
memory usage: 579.8+ KB
None


In [16]:
contacts_conv = contacts_df.copy()

# 1 Id → object (строковый тип)
contacts_conv["Id"] = contacts_conv["Id"].astype(str)

# 2 Created Time → дата и время (дд.мм.гггг чч:мм)
contacts_conv["Created Time"] = pd.to_datetime(
    contacts_conv["Created Time"],
    format="%d.%m.%Y %H:%M",
    errors="coerce"
)

# 3 Modified Time → дата и время (дд.мм.гггг чч:мм)
contacts_conv["Modified Time"] = pd.to_datetime(
    contacts_conv["Modified Time"],
    format="%d.%m.%Y %H:%M",
    errors="coerce"
)

# Проверим результат
print("✅ Типы данных после преобразования:")
print(contacts_conv.dtypes)

✅ Типы данных после преобразования:
Id                            object
Contact Owner Name            object
Created Time          datetime64[ns]
Modified Time         datetime64[ns]
dtype: object


In [17]:
num_duplicates3 = contacts_conv.duplicated().sum()
print(f"Количество полных дубликатов: {num_duplicates3}")

Количество полных дубликатов: 0


In [18]:
# Сохраняем очищенный датафрейм в новый Excel-файл
contacts_conv.to_excel("Contacts_f_clean.xlsx", index=False)

print("✅ Файл успешно сохранён как 'Contacts_f_clean.xlsx'")

✅ Файл успешно сохранён как 'Contacts_f_clean.xlsx'


Теперь Calls

In [19]:
calls_df = pd.read_excel("Calls.xlsx")

# Создаем копию, чтобы не портить оригинал
calls_converted = calls_df.copy()

# 1️⃣ Id → object
calls_converted["Id"] = calls_converted["Id"].astype(str)

# 2️⃣ Call Start Time → datetime (формат 30.06.2023 08:43)
calls_converted["Call Start Time"] = pd.to_datetime(
    calls_converted["Call Start Time"],
    format="%d.%m.%Y %H:%M",
    errors="coerce"
)

# 3️⃣ CONTACTID → object (учитывая пропуски)
calls_converted["CONTACTID"] = (
    calls_converted["CONTACTID"]
    .apply(lambda x: str(int(x)) if pd.notna(x) else np.nan)
)



# Удаляем Dialled Number и Tag
calls_converted = calls_converted.drop(columns=["Dialled Number", "Tag"])

# Проверим результат
print("✅ Типы данных после преобразования:")
print(calls_converted.dtypes)



✅ Типы данных после преобразования:
Id                                    object
Call Start Time               datetime64[ns]
Call Owner Name                       object
CONTACTID                             object
Call Type                             object
Call Duration (in seconds)           float64
Call Status                           object
Outgoing Call Status                  object
Scheduled in CRM                     float64
dtype: object


In [20]:
calls_converted["Scheduled in CRM"].sum()



np.float64(142.0)

In [21]:
num_duplicates4 = calls_converted.duplicated().sum()
print(f"Количество полных дубликатов: {num_duplicates4}")

Количество полных дубликатов: 0


In [22]:
# Сохраняем очищенный датафрейм в новый Excel-файл
calls_converted.to_excel("Calls_f_clean.xlsx", index=False)

print("✅ Файл успешно сохранён как 'Calls_f_clean.xlsx'")

✅ Файл успешно сохранён как 'Calls_f_clean.xlsx'
